In [ ]:
!nvidia-smi
!pip -q install kagglehub


In [ ]:
# 登录 kaggle
import kagglehub
kagglehub.login()


In [ ]:
# 下载 10k 数据集
import kagglehub
path = kagglehub.dataset_download("yuanfangshang/yolo-datasets-10k-yolov8")
print("Path to dataset files:", path)


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/tuanziya666/yolov8s_kuangjing.git"
REPO_DIR = Path("/content/yolov8s_kuangjing")

if REPO_DIR.exists():
    %cd /content/yolov8s_kuangjing
    !git fetch origin
    !git reset --hard origin/main
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd /content/yolov8s_kuangjing

!python -m pip uninstall -y ultralytics >/dev/null 2>&1 || true
!python -m pip install -q -e .


In [ ]:
from pathlib import Path

DATASET_ROOT = Path(path)
yaml_text = f"""path: {DATASET_ROOT}
train: images/train
val: images/val
test: images/test

names:
  0: chuck
  1: gripper
  2: drill_pipe
  3: coal_miner
"""

cfg_path = Path('/content/yolov8s_kuangjing/ultralytics/cfg/datasets/coalmine4_colab.yaml')
cfg_path.write_text(yaml_text, encoding='utf-8')
print(cfg_path.read_text(encoding='utf-8'))


In [ ]:
# 可选：清理旧结果
!rm -rf /content/runs/colab
!rm -rf /content/evals
!rm -f /content/*_colab_results.zip


In [ ]:
from pathlib import Path

assets_dir = Path("/content/yolov8s_kuangjing/ultralytics/assets")
assets_dir.mkdir(parents=True, exist_ok=True)

!wget -q -O {assets_dir / "bus.jpg"} https://raw.githubusercontent.com/tuanziya666/yolov8s_kuangjing/main/ultralytics/assets/bus.jpg
!wget -q -O {assets_dir / "zidane.jpg"} https://raw.githubusercontent.com/tuanziya666/yolov8s_kuangjing/main/ultralytics/assets/zidane.jpg

print((assets_dir / "bus.jpg").exists(), (assets_dir / "zidane.jpg").exists())


In [ ]:
%cd /content/yolov8s_kuangjing

import random
import numpy as np
import torch
from pathlib import Path
from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DATA_CFG = "ultralytics/cfg/datasets/coalmine4_colab.yaml"
EPOCHS = 100
IMGSZ = 640
BATCH = 32
WORKERS = 2
DEVICE = 0
PROJECT = "/content/runs/colab"
VAL_BATCH = 32
TEST_BATCH = 16

MODULE_CONFIGS = {
    "asff": "ultralytics/cfg/models/v8/yolov8s_asff.yaml",
    "carafe": "ultralytics/cfg/models/v8/yolov8s_carafe.yaml",
    "repgfpn": "ultralytics/cfg/models/v8/yolov8s_repgfpn.yaml",
    "dcnv3": "ultralytics/cfg/models/v8/yolov8s_dcnv3.yaml",
    "strip_pooling": "ultralytics/cfg/models/v8/yolov8s_strip_pooling.yaml",
    "coordatt": "ultralytics/cfg/models/v8/yolov8s_coordatt.yaml",
}

for k, v in MODULE_CONFIGS.items():
    print(k, Path(v).exists(), v)

def train_variant(key: str, run_name: str):
    cfg = MODULE_CONFIGS[key]
    print(f"Training {key} with {cfg}")
    model = YOLO(cfg)
    return model.train(
        data=DATA_CFG,
        task='detect',
        project=PROJECT,
        name=run_name,
        device=DEVICE,
        epochs=EPOCHS,
        batch=BATCH,
        imgsz=IMGSZ,
        workers=WORKERS,
        seed=SEED,
        deterministic=True,
        pretrained='yolov8s.pt',
        save=True,
        val=True,
        plots=True,
        verbose=True,
        exist_ok=False,
    )


In [ ]:
# ASFF 实验：只运行这一格
%cd /content/yolov8s_kuangjing

RUN_NAME = "yolov8s_asff_colab"
results = train_variant("asff", RUN_NAME)
results


In [ ]:
# CARAFE 实验：只运行这一格
%cd /content/yolov8s_kuangjing

RUN_NAME = "yolov8s_carafe_colab"
results = train_variant("carafe", RUN_NAME)
results


In [ ]:
# RepGFPN 实验：只运行这一格
%cd /content/yolov8s_kuangjing

RUN_NAME = "yolov8s_repgfpn_colab"
results = train_variant("repgfpn", RUN_NAME)
results


In [ ]:
# DCNv3 实验：只运行这一格
%cd /content/yolov8s_kuangjing

from torchvision.ops import DeformConv2d
print('DeformConv2d ready:', DeformConv2d)

RUN_NAME = "yolov8s_dcnv3_colab"
results = train_variant("dcnv3", RUN_NAME)
results


In [ ]:
# Strip Pooling 实验：只运行这一格
%cd /content/yolov8s_kuangjing

RUN_NAME = "yolov8s_strip_pooling_colab"
results = train_variant("strip_pooling", RUN_NAME)
results


In [ ]:
# Coordinate Attention 实验：只运行这一格
%cd /content/yolov8s_kuangjing

RUN_NAME = "yolov8s_coordatt_colab"
results = train_variant("coordatt", RUN_NAME)
results


In [ ]:
# 评估单元：把 SELECTED_RUN_NAME 改成你刚跑完的那个
%cd /content/yolov8s_kuangjing

from ultralytics import YOLO

SELECTED_RUN_NAME = 'yolov8s_asff_colab'
BEST = f"/content/runs/colab/{SELECTED_RUN_NAME}/weights/best.pt"
print("BEST =", BEST)

model = YOLO(BEST)

print('===== VALIDATE ON VAL =====')
model.val(
    data=DATA_CFG,
    split='val',
    imgsz=IMGSZ,
    batch=VAL_BATCH,
    device=DEVICE,
    plots=True,
    project='/content/evals',
    name=f'{SELECTED_RUN_NAME}_val',
    exist_ok=True,
)

print('===== VALIDATE ON TEST =====')
model.val(
    data=DATA_CFG,
    split='test',
    imgsz=IMGSZ,
    batch=TEST_BATCH,
    device=DEVICE,
    plots=True,
    project='/content/evals',
    name=f'{SELECTED_RUN_NAME}_test',
    exist_ok=True,
)


In [ ]:
# 打包单元：把 SELECTED_RUN_NAME 改成你刚跑完的那个
%cd /content

SELECTED_RUN_NAME = 'yolov8s_asff_colab'
!zip -qr {SELECTED_RUN_NAME}_results.zip /content/runs/colab/{SELECTED_RUN_NAME} /content/evals/{SELECTED_RUN_NAME}_val /content/evals/{SELECTED_RUN_NAME}_test
!ls -lh {SELECTED_RUN_NAME}_results.zip
